# Анализ A/B-платформы

Код выполняется в Docker. Ноутбуки сохраняются в папке `notebooks` на Mac. Доступы поступают из окружения контейнера; пароли не нужно записывать в ноутбук.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import clickhouse_connect
from sqlalchemy import create_engine, URL, text

ch = clickhouse_connect.get_client(
    host=os.environ['CH_HOST'],
    port=int(os.environ['CH_PORT']),
    username=os.environ['CH_USER'],
    password=os.environ['CH_PASSWORD'],
    database='ab',
)
pg = create_engine(URL.create(
    'postgresql+psycopg',
    username=os.environ['PGUSER'],
    password=os.environ['PGPASSWORD'],
    host=os.environ['PGHOST'],
    port=int(os.environ['PGPORT']),
    database=os.environ['PGDATABASE'],
), pool_pre_ping=True)

print('ClickHouse:', ch.query('SELECT 1').result_rows)
with pg.connect() as conn:
    print('PostgreSQL:', conn.execute(text('SELECT 1')).scalar())


## ClickHouse → pandas

In [ ]:
events = ch.query_df('SELECT * FROM ab.events LIMIT 1000')
events.head()

## PostgreSQL → pandas

In [ ]:
with pg.connect() as conn:
    users = pd.read_sql(text('SELECT * FROM core.users LIMIT 1000'), conn)
users.head()

## Эксперименты

In [ ]:
with pg.connect() as conn:
    experiments = pd.read_sql(text('SELECT * FROM ab.experiments LIMIT 100'), conn)
experiments